# Hadith RAG Kaggle - version corrigee complete

Version corrigee : re-ranker BGE via Transformers + merge RAGAS robuste quand `to_pandas()` ne garde pas les colonnes metadata.

Execution conseillee : active Internet + GPU T4. Pour un test rapide, ajoute une cellule avant execution avec `import os; os.environ["MAX_HADITHS"]="5000"; os.environ["EVAL_LIMIT"]="4"`.


In [6]:
import os
os.environ["RAGAS_TIMEOUT"] = "600"
os.environ["RAGAS_MAX_WORKERS"] = "1"
os.environ["RAGAS_MAX_RETRIES"] = "1"
os.environ["RAGAS_MAX_NEW_TOKENS"] = "160"

In [7]:
"""
Version Kaggle complete et corrigee du projet Hadith RAG.

A coller dans un notebook Kaggle ou a uploader comme script.

Ce code couvre exactement :
1. Dataset Hadith : gurgutan/sunnah_ar_en_dataset
2. ChromaDB + similarite cosinus
3. Embeddings : BAAI/bge-m3
4. Vrai re-ranker cross-encoder : BAAI/bge-reranker-v2-m3
5. LLM reel en local Kaggle via HuggingFace Transformers : Qwen/Qwen2.5-0.5B-Instruct
6. Ablations exactes : Generation only vs RAG sans re-ranker vs RAG + re-ranker
7. Evaluation RAGAS reelle, sans metriques proxy
8. Justifications des choix : embeddings, chunking, similarite, LLM, architecture, metriques
9. Questions/prompts arabes corriges en Unicode lisible

Conseil Kaggle :
- Active GPU T4.
- Active Internet.
- Pour un premier test rapide : MAX_HADITHS=5000 et EVAL_LIMIT=4.
- Pour la version finale : MAX_HADITHS=0 et EVAL_LIMIT=10.
"""

from __future__ import annotations

import contextlib
import gc
import io
import json
import os
import random
import re
import subprocess
import sys
import time
import unicodedata
from collections import defaultdict
from pathlib import Path
from typing import Any


In [8]:
# ============================================================
# 1. INSTALLATION DES DEPENDANCES KAGGLE


In [9]:
# ============================================================

INSTALL_DEPS = os.environ.get("INSTALL_DEPS", "1") == "1"

if INSTALL_DEPS:
    packages = [
        "FlagEmbedding",
        "chromadb",
        "datasets",
        "ragas",
        "langchain-huggingface",
        "sentence-transformers",
        "transformers",
        "accelerate",
        "sentencepiece",
        "protobuf",
        "scikit-learn",
        "pandas",
        "numpy",
        "matplotlib",
        "seaborn",
        "tqdm",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


In [10]:
# ============================================================
# 2. IMPORTS ET CONFIGURATION


In [11]:
# ============================================================

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

import chromadb
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, load_dataset


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATASET_NAME = "gurgutan/sunnah_ar_en_dataset"

# 0 = corpus complet. Mets 5000 pour un premier test rapide.
MAX_HADITHS = int(os.environ.get("MAX_HADITHS", "0"))
EVAL_LIMIT = int(os.environ.get("EVAL_LIMIT", "10"))

EMBED_MODEL_NAME = os.environ.get("EMBED_MODEL_NAME", "BAAI/bge-m3")
RERANK_MODEL_NAME = os.environ.get("RERANK_MODEL_NAME", "BAAI/bge-reranker-v2-m3")

# LLM reel local dans Kaggle. 0.5B est choisi pour tenir facilement sur T4.
# Pour une meilleure generation si tu as assez de GPU : Qwen/Qwen2.5-1.5B-Instruct
HF_LLM_MODEL = os.environ.get("HF_LLM_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")

# Embedding d'evaluation RAGAS, leger et multilingue pour eviter OOM.
RAGAS_EVAL_EMBED_MODEL = os.environ.get(
    "RAGAS_EVAL_EMBED_MODEL",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
)

CHROMA_DIR = str(WORK_DIR / "chroma_hadith_bge_m3_cosine")
COLLECTION_NAME = os.environ.get("COLLECTION_NAME", f"hadith_bge_m3_{MAX_HADITHS or 'full'}")
FORCE_REINDEX = os.environ.get("FORCE_REINDEX", "0") == "1"

# Retrieval large : on garde tous les candidats pertinents dans ce top_k selon un seuil.
TOP_K_RETRIEVAL = int(os.environ.get("TOP_K_RETRIEVAL", "50"))
MIN_RETRIEVAL_SIMILARITY = float(os.environ.get("MIN_RETRIEVAL_SIMILARITY", "0.18"))

RERANK_CANDIDATES = int(os.environ.get("RERANK_CANDIDATES", "12"))
TOP_CONTEXTS_FOR_LLM = int(os.environ.get("TOP_CONTEXTS_FOR_LLM", "5"))
MAX_CONTEXT_CHARS = int(os.environ.get("MAX_CONTEXT_CHARS", "12000"))

EMBED_BATCH_SIZE = int(os.environ.get("EMBED_BATCH_SIZE", "24"))
INDEX_BATCH_SIZE = int(os.environ.get("INDEX_BATCH_SIZE", "96"))
RERANK_BATCH_SIZE = int(os.environ.get("RERANK_BATCH_SIZE", "2"))
RERANK_MAX_LENGTH = int(os.environ.get("RERANK_MAX_LENGTH", "384"))
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "360"))
RAGAS_MAX_NEW_TOKENS = int(os.environ.get("RAGAS_MAX_NEW_TOKENS", "160"))
RAGAS_TIMEOUT = int(os.environ.get("RAGAS_TIMEOUT", "600"))
RAGAS_MAX_WORKERS = int(os.environ.get("RAGAS_MAX_WORKERS", "1"))
RAGAS_MAX_RETRIES = int(os.environ.get("RAGAS_MAX_RETRIES", "1"))

QUIET_MODELS = os.environ.get("QUIET_MODELS", "1") == "1"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"WORK_DIR={WORK_DIR}")
print(f"DEVICE={DEVICE}")
print(f"MAX_HADITHS={MAX_HADITHS}, EVAL_LIMIT={EVAL_LIMIT}")
print(f"Embedding={EMBED_MODEL_NAME}")
print(f"Reranker={RERANK_MODEL_NAME}")
print(f"LLM={HF_LLM_MODEL}")


WORK_DIR=/kaggle/working
DEVICE=cuda
MAX_HADITHS=0, EVAL_LIMIT=10
Embedding=BAAI/bge-m3
Reranker=BAAI/bge-reranker-v2-m3
LLM=Qwen/Qwen2.5-0.5B-Instruct


In [12]:
# ============================================================
# 3. JUSTIFICATIONS A METTRE DANS LE RAPPORT


In [13]:
# ============================================================

JUSTIFICATION_TEXT = """
# Justification des choix techniques

## Architecture
L'architecture utilise un pipeline RAG en trois etapes : recuperation dense dans ChromaDB,
reclassement par cross-encoder, puis generation par un LLM local. Cette separation est importante
pour un corpus religieux : le modele ne doit pas inventer des hadiths, il doit d'abord recuperer
les textes candidats, puis produire une reponse fondee sur les sources.

## Embeddings : BAAI/bge-m3
BAAI/bge-m3 est choisi parce qu'il est multilingue et adapte a la recherche semantique dense.
Le dataset contient de l'arabe et de l'anglais ; BGE-M3 permet donc de traiter les requetes dans
les deux langues avec un seul espace vectoriel.

## Methode de chunking
Le chunk est un hadith complet avec ses metadonnees : livre, chapitre, narrateur, texte arabe et
texte anglais. Ce choix respecte l'unite naturelle du corpus. Decouper un hadith en petits morceaux
risquerait de separer le texte de ses metadonnees et de rendre les citations moins fiables.

## Similarite cosinus
La similarite cosinus est utilisee dans ChromaDB avec `hnsw:space=cosine`. Elle est adaptee aux
embeddings normalises car elle compare l'orientation semantique des vecteurs plutot que leur norme.
C'est une mesure standard pour la recherche dense.

## Re-ranker : BAAI/bge-reranker-v2-m3
La recuperation vectorielle donne des candidats pertinents mais peut mal classer certains textes.
Le re-ranker cross-encoder lit la paire (requete, hadith candidat) et donne un score de pertinence
plus precis. Il est applique uniquement sur les meilleurs candidats pour garder un cout raisonnable.
Le mode attendu dans les resultats est `cross_encoder_plus_hybrid`.

## LLM : Qwen/Qwen2.5-0.5B-Instruct sur Kaggle
Ollama n'est pas pratique sur Kaggle car il faut un serveur local separe. La version Kaggle utilise
donc un LLM HuggingFace local, charge dans le notebook. Qwen2.5-0.5B-Instruct est choisi parce qu'il
est instruction-tuned, multilingue, compatible avec GPU T4, et permet une execution reproductible
sans API externe. Pour une meilleure qualite, on peut remplacer par Qwen2.5-1.5B-Instruct.

## Metriques RAGAS
RAGAS est utilise pour evaluer les reponses et les contextes. Les metriques principales sont :
faithfulness pour verifier si la reponse est soutenue par les contextes, context_recall pour mesurer
si les contextes couvrent la reference, context_precision pour evaluer le classement des contextes,
answer_relevancy pour mesurer la pertinence de la reponse, et answer_correctness pour comparer la
reponse a la reference. Les metriques de contexte sont appliquees aux deux modes RAG ; pour
Generation only, les metriques de reponse sont les plus pertinentes car il n'y a pas de contexte.
""".strip()


def save_justification() -> None:
    path = WORK_DIR / "justification_choix_architecture_metriques.md"
    path.write_text(JUSTIFICATION_TEXT, encoding="utf-8")
    print(f"Justification sauvegardee : {path}")


In [15]:
# ============================================================

AR_NORMALIZE_TABLE = str.maketrans({
    "أ": "ا",
    "إ": "ا",
    "آ": "ا",
    "ٱ": "ا",
    "ى": "ي",
    "ة": "ه",
    "ؤ": "و",
    "ئ": "ي",
})

STOPWORDS = {
    "the", "and", "or", "to", "of", "in", "a", "an", "is", "are", "was", "were",
    "what", "how", "did", "does", "about", "with", "from", "for", "their", "his",
    "على", "عن", "في", "ما", "هي", "هو", "من", "الى", "إلى", "قال", "النبي",
    "رسول", "الله", "صلى", "عليه", "وسلم",
}

AR_TO_EN_TERMS = {
    "الصيام": "fasting sawm ramadan",
    "صيام": "fasting sawm ramadan",
    "الصوم": "fasting sawm ramadan",
    "صوم": "fasting sawm ramadan",
    "رمضان": "ramadan fasting forgiveness reward",
    "الصلاة": "prayer salat salah worship",
    "صلاة": "prayer salat salah worship",
    "الزكاة": "zakat almsgiving charity",
    "زكاة": "zakat almsgiving charity",
    "الحج": "hajj pilgrimage",
    "حج": "hajj pilgrimage",
    "الشهادة": "shahada testimony",
    "أركان": "pillars islam five",
    "اركان": "pillars islam five",
    "الأخلاق": "ethics morals character",
    "الصدق": "truthfulness honesty",
    "صدق": "truthfulness honesty",
    "الأمانة": "trust honesty amanah",
    "امانة": "trust honesty amanah",
    "الوالدين": "parents mother father kindness",
    "والدين": "parents mother father kindness",
    "بر": "kindness honoring parents",
    "الجيران": "neighbors neighbour kindness",
    "جار": "neighbor",
    "النفاق": "hypocrisy hypocrite lies promise trust",
    "المنافق": "hypocrite lies promise trust",
    "فضائل": "virtues merits reward",
    "فضل": "virtue merit reward",
    "حكم": "ruling obligation required",
}

EN_SYNONYMS = {
    "prayer": "salat salah worship",
    "salat": "prayer salah worship",
    "salah": "prayer salat worship",
    "fasting": "sawm ramadan forgiveness reward",
    "ramadan": "fasting sawm forgiveness reward",
    "parents": "mother father honoring kindness",
    "mother": "parents honoring kindness",
    "father": "parents honoring kindness",
    "neighbors": "neighbour neighbor kindness",
    "neighbor": "neighbour neighbor kindness",
    "hypocrite": "munafiq lies promise trust",
    "truth": "truthfulness honesty amanah",
    "truthfulness": "honesty amanah",
    "charity": "sadaqah zakat",
    "pillars": "five islam shahada prayer zakat fasting hajj",
}


def is_arabic(text: str) -> bool:
    text = str(text or "")
    arabic_chars = sum(1 for c in text if "\u0600" <= c <= "\u06FF")
    return arabic_chars > max(1, len(text)) * 0.25


def normalize_text(text: str) -> str:
    text = str(text or "").lower()
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = text.translate(AR_NORMALIZE_TABLE)
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def score_tokens(text: str) -> set[str]:
    return {t for t in normalize_text(text).split() if len(t) > 2 and t not in STOPWORDS}


def expand_query(query: str) -> str:
    expanded = str(query)
    if is_arabic(query):
        for ar_term, en_terms in AR_TO_EN_TERMS.items():
            if ar_term in query:
                expanded += " " + en_terms
    else:
        q_lower = query.lower()
        for term, syns in EN_SYNONYMS.items():
            if term in q_lower:
                expanded += " " + syns
    return expanded


def clear_memory() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def l2_normalize(vectors: np.ndarray) -> np.ndarray:
    arr = np.asarray(vectors, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    return arr / np.maximum(norms, 1e-12)


In [16]:
# ============================================================
# 5. DATASET HADITH ET CHUNKING


In [17]:
# ============================================================

def load_hadith_dataset() -> pd.DataFrame:
    print(f"Chargement dataset : {DATASET_NAME}")
    ds = load_dataset(DATASET_NAME)
    split = list(ds.keys())[0]
    df_raw = ds[split].to_pandas()

    required_cols = [
        "hadith_uid",
        "hadith_text_en",
        "hadith_text_ar",
        "hadith_narrator_en",
        "hadith_chapter_name_en",
        "hadith_chapter_name_ar",
        "book_title_en",
    ]
    missing = [c for c in required_cols if c not in df_raw.columns]
    if missing:
        raise ValueError(f"Colonnes manquantes : {missing}")

    df = df_raw.copy()
    for col in required_cols:
        df[col] = df[col].fillna("").astype(str).str.strip()

    df = df[
        (df["hadith_text_en"].str.len() > 10)
        | (df["hadith_text_ar"].str.len() > 10)
    ].reset_index(drop=True)

    if MAX_HADITHS > 0 and len(df) > MAX_HADITHS:
        df = df.sample(n=MAX_HADITHS, random_state=SEED).sort_index().reset_index(drop=True)

    print(f"Hadiths indexes : {len(df):,}")
    print(f"Livres          : {df['book_title_en'].nunique():,}")
    print(f"Chapitres       : {df['hadith_chapter_name_en'].nunique():,}")
    return df


def build_hadith_chunks(df: pd.DataFrame) -> list[dict[str, Any]]:
    chunks = []
    for row in df.itertuples(index=False):
        uid = str(row.hadith_uid)
        text_combined = (
            f"Book: {row.book_title_en}\n"
            f"Chapter EN: {row.hadith_chapter_name_en}\n"
            f"Chapter AR: {row.hadith_chapter_name_ar}\n"
            f"Narrator: {row.hadith_narrator_en}\n"
            f"English Hadith: {row.hadith_text_en}\n"
            f"Arabic Hadith: {row.hadith_text_ar}"
        ).strip()
        chunks.append({
            "chunk_id": f"h__{uid}",
            "uid": uid,
            "book": str(row.book_title_en),
            "chapter_en": str(row.hadith_chapter_name_en),
            "chapter_ar": str(row.hadith_chapter_name_ar),
            "narrator": str(row.hadith_narrator_en),
            "text_en": str(row.hadith_text_en),
            "text_ar": str(row.hadith_text_ar),
            "text_combined": text_combined,
        })
    return chunks


def build_chapter_graph(chunks: list[dict[str, Any]]):
    graph = nx.Graph()
    chapter_to_uids: dict[str, list[str]] = defaultdict(list)
    uid_to_chapter: dict[str, str] = {}
    for c in chunks:
        chapter_key = f"{c['book']}::{c['chapter_en']}"
        hadith_node = f"hadith::{c['uid']}"
        chapter_node = f"chapter::{chapter_key}"
        graph.add_node(hadith_node, node_type="hadith", uid=c["uid"])
        graph.add_node(chapter_node, node_type="chapter", chapter=chapter_key)
        graph.add_edge(hadith_node, chapter_node, weight=1.0)
        chapter_to_uids[chapter_key].append(c["uid"])
        uid_to_chapter[c["uid"]] = chapter_key
    return graph, chapter_to_uids, uid_to_chapter


def save_data_diagnostics(df: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(df["hadith_text_en"].str.len().clip(0, 1400), bins=60, color="#2563eb")
    axes[0].set_title("Longueur des hadiths EN")
    axes[0].set_xlabel("Caracteres")
    axes[0].set_ylabel("Frequence")
    axes[1].hist(df["hadith_text_ar"].str.len().clip(0, 1800), bins=60, color="#ea580c")
    axes[1].set_title("Longueur des hadiths AR")
    axes[1].set_xlabel("Caracteres")
    plt.tight_layout()
    out = WORK_DIR / "data_distribution.png"
    plt.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Diagnostic data sauvegarde : {out}")


In [18]:
# ============================================================
# 6. EMBEDDINGS BGE-M3 ET CHROMADB


In [19]:
# ============================================================

def load_embedding_model():
    from FlagEmbedding import BGEM3FlagModel

    use_fp16 = torch.cuda.is_available()
    print(f"Chargement embedding model : {EMBED_MODEL_NAME} | fp16={use_fp16}")
    return BGEM3FlagModel(EMBED_MODEL_NAME, use_fp16=use_fp16)


def bge_dense_encode(embed_model, texts: str | list[str], batch_size: int = EMBED_BATCH_SIZE) -> np.ndarray:
    single = isinstance(texts, str)
    text_list = [texts] if single else list(texts)
    if not text_list:
        return np.empty((0, 0), dtype=np.float32)

    stdout_buf, stderr_buf = io.StringIO(), io.StringIO()
    redirect = contextlib.redirect_stdout(stdout_buf) if QUIET_MODELS else contextlib.nullcontext()
    redirect_err = contextlib.redirect_stderr(stderr_buf) if QUIET_MODELS else contextlib.nullcontext()

    with redirect, redirect_err:
        try:
            output = embed_model.encode(text_list, return_dense=True, batch_size=batch_size)
        except TypeError:
            output = embed_model.encode(text_list, return_dense=True)

    vectors = output["dense_vecs"] if isinstance(output, dict) else output
    vectors = l2_normalize(np.asarray(vectors, dtype=np.float32))
    return vectors[0] if single else vectors


def get_or_build_chroma_collection(embed_model, chunks: list[dict[str, Any]]):
    client = chromadb.PersistentClient(path=CHROMA_DIR)

    if FORCE_REINDEX:
        with contextlib.suppress(Exception):
            client.delete_collection(COLLECTION_NAME)

    try:
        collection = client.get_collection(COLLECTION_NAME)
        print(f"Collection Chroma existante : {collection.count():,}/{len(chunks):,}")
    except Exception:
        collection = client.create_collection(
            name=COLLECTION_NAME,
            metadata={"hnsw:space": "cosine"},
        )
        print("Nouvelle collection Chroma creee avec hnsw:space=cosine.")

    existing_ids = set()
    if collection.count() > 0:
        try:
            existing_ids = set(collection.get(include=[])["ids"])
        except Exception:
            existing_ids = set()

    pending = [c for c in chunks if c["chunk_id"] not in existing_ids]
    if not pending:
        print("Index Chroma deja complet.")
        return collection

    print(f"Indexation Chroma : {len(pending):,} chunks.")
    for start in range(0, len(pending), INDEX_BATCH_SIZE):
        batch = pending[start:start + INDEX_BATCH_SIZE]
        ids = [c["chunk_id"] for c in batch]
        documents = [c["text_combined"] for c in batch]
        metadatas = [
            {
                "uid": c["uid"],
                "book": c["book"],
                "chapter_en": c["chapter_en"],
                "chapter_ar": c["chapter_ar"],
                "narrator": c["narrator"],
            }
            for c in batch
        ]
        embeddings = bge_dense_encode(embed_model, documents).tolist()
        collection.add(ids=ids, documents=documents, metadatas=metadatas, embeddings=embeddings)

        done = min(start + INDEX_BATCH_SIZE, len(pending))
        if done == len(pending) or done % (INDEX_BATCH_SIZE * 25) == 0:
            print(f"  {done:,}/{len(pending):,} chunks indexes")
        del embeddings
        clear_memory()

    print(f"Index final : {collection.count():,}")
    return collection


In [20]:
# ============================================================
# 7. RETRIEVAL ET IDF LOCAL


In [21]:
# ============================================================

def make_token_idf(chunks: list[dict[str, Any]]):
    df_counts: dict[str, int] = defaultdict(int)
    for c in chunks:
        for token in score_tokens(c["text_combined"]):
            df_counts[token] += 1
    n_docs = len(chunks)

    def idf(token: str) -> float:
        return float(np.log((n_docs + 1) / (df_counts.get(token, 0) + 1)) + 1.0)

    return idf


def weighted_overlap(query_terms: set[str], doc_terms: set[str], idf_func) -> float:
    if not query_terms:
        return 0.0
    denom = sum(idf_func(t) for t in query_terms)
    if denom <= 0:
        return 0.0
    return sum(idf_func(t) for t in query_terms & doc_terms) / denom


def lexical_score(query: str, h: dict[str, Any], idf_func) -> float:
    q_terms = score_tokens(expand_query(query))
    doc_terms = score_tokens(
        f"{h.get('book', '')} {h.get('chapter_en', '')} {h.get('chapter_ar', '')} "
        f"{h.get('narrator', '')} {h.get('text_en', '')} {h.get('text_ar', '')}"
    )
    return weighted_overlap(q_terms, doc_terms, idf_func)


def retrieve_related_hadiths(
    query: str,
    embed_model,
    collection,
    chunk_map: dict[str, dict[str, Any]],
    chunk_id_to_uid: dict[str, str],
    top_k: int = TOP_K_RETRIEVAL,
    min_similarity: float = MIN_RETRIEVAL_SIMILARITY,
) -> list[dict[str, Any]]:
    q_expanded = expand_query(query)
    q_embedding = bge_dense_encode(embed_model, q_expanded).reshape(-1).tolist()

    result = collection.query(
        query_embeddings=[q_embedding],
        n_results=top_k,
        include=["metadatas", "documents", "distances"],
    )

    candidates = []
    for rank, chunk_id in enumerate(result["ids"][0], 1):
        uid = chunk_id_to_uid.get(chunk_id) or result["metadatas"][0][rank - 1].get("uid")
        if uid not in chunk_map:
            continue
        distance = float(result["distances"][0][rank - 1])
        similarity = 1.0 - distance
        if similarity < min_similarity:
            continue
        item = chunk_map[uid].copy()
        item["rank_retrieval"] = rank
        item["score_retrieval"] = round(similarity, 4)
        item["source_stage"] = "chroma_vector"
        candidates.append(item)
    return candidates


In [22]:
# ============================================================
# 8. VRAI RE-RANKER CROSS-ENCODER


In [23]:
# ============================================================

def load_cross_encoder_reranker():
    """Charge le vrai modele BGE reranker, mais avec Transformers.

    Sur Kaggle, `FlagReranker.compute_score()` peut rester bloque a
    `pre tokenize: 0/...`. Cette version utilise le meme modele
    BAAI/bge-reranker-v2-m3 comme cross-encoder, mais avec un passage
    manuel tokenizer + model, plus stable dans les notebooks Kaggle.
    """
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    print(f"Chargement re-ranker HF cross-encoder : {RERANK_MODEL_NAME} | dtype={dtype}")
    tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL_NAME, trust_remote_code=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        RERANK_MODEL_NAME,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    model = model.to(DEVICE)
    model.eval()
    print("Re-ranker cross-encoder charge avec Transformers.")
    return {"tokenizer": tokenizer, "model": model, "mode": "hf_cross_encoder_manual"}


def candidate_text_for_reranker(query: str, h: dict[str, Any]) -> str:
    if is_arabic(query):
        primary = h.get("text_ar", "")
        secondary = h.get("text_en", "")
    else:
        primary = h.get("text_en", "")
        secondary = h.get("text_ar", "")
    metadata = f"Book: {h.get('book', '')}\nChapter: {h.get('chapter_en', '')}\nNarrator: {h.get('narrator', '')}"
    return f"{metadata}\n{primary}\n{secondary}"[:1200]


def manual_cross_encoder_scores(query: str, candidates: list[dict[str, Any]], reranker) -> list[float]:
    tokenizer = reranker["tokenizer"]
    model = reranker["model"]
    device = next(model.parameters()).device
    scores: list[float] = []

    for start in range(0, len(candidates), RERANK_BATCH_SIZE):
        batch = candidates[start:start + RERANK_BATCH_SIZE]
        batch_texts = [candidate_text_for_reranker(query, h) for h in batch]
        encoded = tokenizer(
            [query] * len(batch_texts),
            batch_texts,
            padding=True,
            truncation=True,
            max_length=RERANK_MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        with torch.no_grad():
            output = model(**encoded)
            logits = output.logits
            if logits.ndim == 2 and logits.shape[1] > 1:
                batch_scores = logits[:, -1]
            else:
                batch_scores = logits.reshape(-1)
        scores.extend(batch_scores.float().detach().cpu().tolist())
        del encoded, output, logits, batch_scores
        clear_memory()

    return scores


def rag_without_reranker(candidates: list[dict[str, Any]], top_n: int = TOP_CONTEXTS_FOR_LLM) -> list[dict[str, Any]]:
    selected = []
    for idx, h in enumerate(candidates[:top_n], 1):
        item = h.copy()
        item["score_final"] = item.get("score_retrieval", 0.0)
        item["rerank_mode"] = "vector_only_no_reranker"
        item["rank_final"] = idx
        selected.append(item)
    return selected


def rag_with_cross_encoder_reranker(
    query: str,
    candidates: list[dict[str, Any]],
    reranker,
    idf_func,
    top_n: int = TOP_CONTEXTS_FOR_LLM,
) -> list[dict[str, Any]]:
    if not candidates:
        return []

    subset = candidates[:RERANK_CANDIDATES]
    raw_scores = manual_cross_encoder_scores(query, subset, reranker)

    cross_scores = np.asarray(raw_scores, dtype=np.float32).reshape(-1)
    if len(cross_scores) > 1 and float(cross_scores.max()) > float(cross_scores.min()):
        cross_scores = (cross_scores - cross_scores.min()) / (cross_scores.max() - cross_scores.min())
    else:
        cross_scores = np.ones_like(cross_scores) * 0.5

    ranked = []
    for cross_score, h in zip(cross_scores.tolist(), subset):
        retrieval = float(h.get("score_retrieval", 0.0))
        lexical = lexical_score(query, h, idf_func)
        final = 0.78 * float(cross_score) + 0.15 * retrieval + 0.07 * lexical
        item = h.copy()
        item["score_cross_encoder"] = round(float(cross_score), 4)
        item["score_lexical"] = round(float(lexical), 4)
        item["score_final"] = round(float(final), 4)
        item["rerank_mode"] = "cross_encoder_plus_hybrid"
        ranked.append(item)

    ranked = sorted(ranked, key=lambda x: x["score_final"], reverse=True)
    for idx, item in enumerate(ranked[:top_n], 1):
        item["rank_final"] = idx
    clear_memory()
    return ranked[:top_n]


In [25]:
# ============================================================

EVAL_DATASET = [
    {
        "question": "What did the Prophet say about the importance of prayer (salat)?",
        "reference": "Prayer is central in Islam and is one of the most important obligations after faith.",
        "language": "en",
    },
    {
        "question": "What are the virtues of fasting in Ramadan?",
        "reference": "Fasting Ramadan with faith and seeking reward is a major act of worship and brings forgiveness.",
        "language": "en",
    },
    {
        "question": "What did the Prophet say about kindness to parents?",
        "reference": "Islam strongly emphasizes honoring and being kind to parents, especially the mother.",
        "language": "en",
    },
    {
        "question": "How should a Muslim treat their neighbors?",
        "reference": "A Muslim should treat neighbors well, protect them from harm, and show kindness.",
        "language": "en",
    },
    {
        "question": "What are the signs of a hypocrite?",
        "reference": "Signs include lying when speaking, breaking promises, and betraying trust.",
        "language": "en",
    },
    {
        "question": "ما فضل الصلاة على النبي؟",
        "reference": "الصلاة على النبي لها فضل عظيم، ومن صلى عليه صلاة صلى الله عليه بها عشرا.",
        "language": "ar",
    },
    {
        "question": "ما حكم الصيام في رمضان؟",
        "reference": "صيام رمضان فريضة وركن من أركان الإسلام.",
        "language": "ar",
    },
    {
        "question": "ماذا قال النبي عن بر الوالدين؟",
        "reference": "بر الوالدين من أعظم الأعمال، وفيه تأكيد على حق الأم والأب وحسن صحبتهما.",
        "language": "ar",
    },
    {
        "question": "ما هي أركان الإسلام الخمسة؟",
        "reference": "أركان الإسلام خمسة: الشهادتان، الصلاة، الزكاة، الصوم، والحج.",
        "language": "ar",
    },
    {
        "question": "ماذا قال النبي عن الصدق والأمانة؟",
        "reference": "الصدق يهدي إلى البر، والأمانة من صفات المؤمن، وخيانة الأمانة من علامات النفاق.",
        "language": "ar",
    },
][:EVAL_LIMIT]


In [27]:
# ============================================================

def context_texts_for_language(contexts: list[dict[str, Any]], language: str) -> list[str]:
    if language == "ar":
        return [c.get("text_ar", "") for c in contexts]
    return [c.get("text_en", "") for c in contexts]


def compact_evidence(contexts: list[dict[str, Any]]) -> list[dict[str, Any]]:
    evidence = []
    for c in contexts:
        evidence.append({
            "uid": c.get("uid"),
            "book": c.get("book"),
            "chapter_en": c.get("chapter_en"),
            "chapter_ar": c.get("chapter_ar"),
            "narrator": c.get("narrator"),
            "score_retrieval": c.get("score_retrieval"),
            "score_cross_encoder": c.get("score_cross_encoder"),
            "score_final": c.get("score_final"),
            "rerank_mode": c.get("rerank_mode"),
            "text_en": c.get("text_en"),
            "text_ar": c.get("text_ar"),
        })
    return evidence


def prepare_ablation_rows(
    embed_model,
    collection,
    chunk_map: dict[str, dict[str, Any]],
    chunk_id_to_uid: dict[str, str],
    reranker,
    idf_func,
) -> pd.DataFrame:
    rows = []
    all_related_for_export = []

    for idx, item in enumerate(EVAL_DATASET, 1):
        question = item["question"]
        print(f"[Retrieval {idx}/{len(EVAL_DATASET)}] {question}")
        candidates = retrieve_related_hadiths(
            question,
            embed_model=embed_model,
            collection=collection,
            chunk_map=chunk_map,
            chunk_id_to_uid=chunk_id_to_uid,
        )

        all_related_for_export.append({
            "question": question,
            "language": item["language"],
            "n_related_retrieved": len(candidates),
            "related_hadiths": compact_evidence(candidates),
        })

        no_rerank_contexts = rag_without_reranker(candidates)
        reranked_contexts = rag_with_cross_encoder_reranker(question, candidates, reranker, idf_func)

        rows.append({
            "system": "Generation only",
            "question": question,
            "reference": item["reference"],
            "language": item["language"],
            "contexts": [],
            "evidence": [],
            "n_candidates": 0,
            "n_contexts": 0,
            "top_score": np.nan,
            "rerank_mode": "none_generation_only",
        })

        rows.append({
            "system": "RAG sans re-ranker",
            "question": question,
            "reference": item["reference"],
            "language": item["language"],
            "contexts": context_texts_for_language(no_rerank_contexts, item["language"]),
            "evidence": compact_evidence(no_rerank_contexts),
            "n_candidates": len(candidates),
            "n_contexts": len(no_rerank_contexts),
            "top_score": no_rerank_contexts[0]["score_final"] if no_rerank_contexts else np.nan,
            "rerank_mode": "vector_only_no_reranker",
        })

        rows.append({
            "system": "RAG + re-ranker",
            "question": question,
            "reference": item["reference"],
            "language": item["language"],
            "contexts": context_texts_for_language(reranked_contexts, item["language"]),
            "evidence": compact_evidence(reranked_contexts),
            "n_candidates": len(candidates),
            "n_contexts": len(reranked_contexts),
            "top_score": reranked_contexts[0]["score_final"] if reranked_contexts else np.nan,
            "rerank_mode": reranked_contexts[0]["rerank_mode"] if reranked_contexts else "cross_encoder_plus_hybrid",
        })

    related_path = WORK_DIR / "all_related_hadiths_per_question.json"
    related_path.write_text(json.dumps(all_related_for_export, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Hadiths relatifs sauvegardes : {related_path}")

    df_rows = pd.DataFrame(rows)
    expected = df_rows[df_rows["system"] == "RAG + re-ranker"]["rerank_mode"].unique().tolist()
    if "cross_encoder_plus_hybrid" not in expected:
        raise RuntimeError("Le vrai re-ranker n'a pas tourne : cross_encoder_plus_hybrid absent.")

    prep_path = WORK_DIR / "ablation_contexts_before_generation.json"
    prep_path.write_text(json.dumps(rows, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(f"Contextes d'ablation sauvegardes : {prep_path}")
    return df_rows


In [28]:
# ============================================================
# 11. LLM HUGGINGFACE LOCAL POUR KAGGLE


In [29]:
# ============================================================

def load_generation_llm():
    from transformers import AutoModelForCausalLM, AutoTokenizer

    print(f"Chargement LLM local : {HF_LLM_MODEL}")
    tokenizer = AutoTokenizer.from_pretrained(HF_LLM_MODEL, trust_remote_code=True)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        HF_LLM_MODEL,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    if not torch.cuda.is_available():
        model = model.to("cpu")
    model.eval()
    return tokenizer, model


def generate_with_llm(tokenizer, model, system: str, user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_prompt},
    ]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = f"System: {system}\nUser: {user_prompt}\nAssistant:"

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=4096)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(generated, skip_special_tokens=True).strip()
    clear_memory()
    return answer


def build_context_block(row: dict[str, Any]) -> str:
    blocks = []
    total = 0
    for i, ev in enumerate(row.get("evidence", []), 1):
        primary = ev.get("text_ar", "") if row["language"] == "ar" else ev.get("text_en", "")
        other = ev.get("text_en", "") if row["language"] == "ar" else ev.get("text_ar", "")
        block = (
            f"[H{i}]\n"
            f"Book: {ev.get('book')}\n"
            f"Chapter EN: {ev.get('chapter_en')}\n"
            f"Chapter AR: {ev.get('chapter_ar')}\n"
            f"Narrator: {ev.get('narrator')}\n"
            f"Primary text:\n{primary}\n"
            f"Other language text:\n{other}"
        )
        if blocks and total + len(block) > MAX_CONTEXT_CHARS:
            break
        blocks.append(block)
        total += len(block)
    return "\n\n".join(blocks)


def build_prompt_for_row(row: dict[str, Any]) -> tuple[str, str]:
    question = row["question"]
    system_name = row["system"]

    if system_name == "Generation only":
        if row["language"] == "ar":
            system = "أنت مساعد يجيب باختصار عن أسئلة الحديث. لا تخترع أرقام أحاديث أو مصادر."
            prompt = (
                "أجب عن السؤال اعتمادا على معرفة النموذج فقط، بدون استخدام نظام استرجاع.\n"
                "اذكر بوضوح أن هذه إجابة بدون مصادر مسترجعة.\n\n"
                f"السؤال: {question}\n\n"
                "الإجابة:"
            )
        else:
            system = "You answer Hadith-related questions concisely. Do not invent hadith IDs or sources."
            prompt = (
                "Answer using the model's general knowledge only, without retrieved context.\n"
                "Clearly state that this is a generation-only answer without retrieved sources.\n\n"
                f"Question: {question}\n\n"
                "Answer:"
            )
        return system, prompt

    context = build_context_block(row)
    if row["language"] == "ar":
        system = (
            "أنت مساعد RAG متخصص في الأحاديث. استخدم النصوص المسترجعة فقط. "
            "إذا لم تكف النصوص، قل إن الأدلة غير كافية. استشهد بالمعرفات مثل [H1]."
        )
        prompt = (
            "استخدم الأحاديث التالية فقط للإجابة. لا تضف حديثا غير موجود في السياق.\n\n"
            f"{context}\n\n"
            f"السؤال: {question}\n\n"
            "الإجابة مع الاستشهادات:"
        )
    else:
        system = (
            "You are a Hadith RAG assistant. Use only the retrieved hadith evidence. "
            "If the evidence is insufficient, say so. Cite evidence IDs such as [H1]."
        )
        prompt = (
            "Use ONLY the following retrieved hadiths to answer. Do not add hadiths not present in the context.\n\n"
            f"{context}\n\n"
            f"Question: {question}\n\n"
            "Answer with citations:"
        )
    return system, prompt


def generate_answers(ablation_df: pd.DataFrame, tokenizer, model) -> pd.DataFrame:
    rows = []
    for idx, row in enumerate(ablation_df.to_dict("records"), 1):
        print(f"[Generation {idx}/{len(ablation_df)}] {row['system']} | {row['question'][:60]}")
        system, prompt = build_prompt_for_row(row)
        t0 = time.time()
        answer = generate_with_llm(tokenizer, model, system, prompt)
        row["answer"] = answer
        row["seconds_generation"] = round(time.time() - t0, 2)
        rows.append(row)

    results_df = pd.DataFrame(rows)
    csv_df = results_df.copy()
    csv_df["contexts_json"] = csv_df["contexts"].apply(lambda x: json.dumps(x, ensure_ascii=False))
    csv_df["evidence_json"] = csv_df["evidence"].apply(lambda x: json.dumps(x, ensure_ascii=False))
    csv_df = csv_df.drop(columns=["contexts", "evidence"])
    out_csv = WORK_DIR / "ablation_generation_vs_rag_vs_reranker_raw.csv"
    csv_df.to_csv(out_csv, index=False, encoding="utf-8-sig")

    out_json = WORK_DIR / "ablation_generation_vs_rag_vs_reranker_raw.json"
    out_json.write_text(json.dumps(rows, ensure_ascii=False, indent=2, default=str), encoding="utf-8")

    print(f"Resultats bruts CSV : {out_csv}")
    print(f"Resultats bruts JSON : {out_json}")
    return results_df


In [30]:
# ============================================================
# 12. RAGAS REEL AVEC MODELES LOCAUX HUGGINGFACE


In [31]:
# ============================================================

def make_ragas_dataset(rows: list[dict[str, Any]]) -> Dataset:
    data = []
    for i, r in enumerate(rows):
        contexts = r.get("contexts", [])
        if not contexts:
            contexts = [""]
        data.append({
            "row_id": i,
            "system": r["system"],
            "language": r["language"],
            "user_input": r["question"],
            "response": r["answer"],
            "retrieved_contexts": contexts,
            "reference": r["reference"],
            # Alias pour anciennes versions de RAGAS
            "question": r["question"],
            "answer": r["answer"],
            "contexts": contexts,
            "ground_truth": r["reference"],
        })
    return Dataset.from_list(data)


def get_metric_object(metrics_module, *candidate_names: str):
    for name in candidate_names:
        obj = getattr(metrics_module, name, None)
        if obj is None:
            continue
        if isinstance(obj, type):
            return obj()
        return obj
    return None


def build_ragas_metrics(kind: str):
    import ragas.metrics as ragas_metrics

    if kind == "answer":
        requested = [
            ("answer_relevancy", ("ResponseRelevancy", "AnswerRelevancy", "answer_relevancy")),
            ("answer_correctness", ("AnswerCorrectness", "answer_correctness")),
        ]
    elif kind == "rag":
        requested = [
            ("faithfulness", ("Faithfulness", "faithfulness")),
            ("context_recall", ("ContextRecall", "LLMContextRecall", "context_recall")),
            ("context_precision", ("ContextPrecision", "LLMContextPrecisionWithReference", "context_precision")),
        ]
    else:
        raise ValueError(kind)

    metrics = []
    missing = []
    for label, names in requested:
        metric = get_metric_object(ragas_metrics, *names)
        if metric is None:
            missing.append(label)
        else:
            metrics.append(metric)

    if missing:
        print(f"Metriques RAGAS non disponibles dans cette version : {missing}")
    if not metrics:
        raise RuntimeError(f"Aucune metrique RAGAS disponible pour kind={kind}")
    return metrics


def build_ragas_local_wrappers():
    from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
    from ragas.embeddings import LangchainEmbeddingsWrapper
    from ragas.llms import LangchainLLMWrapper
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

    print("Chargement du LLM juge RAGAS local...")
    tokenizer = AutoTokenizer.from_pretrained(HF_LLM_MODEL, trust_remote_code=True)
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        HF_LLM_MODEL,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    if not torch.cuda.is_available():
        model = model.to("cpu")

    text_generation_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=RAGAS_MAX_NEW_TOKENS,
        do_sample=False,
        return_full_text=False,
    )
    evaluator_llm = LangchainLLMWrapper(HuggingFacePipeline(pipeline=text_generation_pipeline))

    print(f"Chargement embeddings RAGAS : {RAGAS_EVAL_EMBED_MODEL}")
    eval_embeddings = HuggingFaceEmbeddings(
        model_name=RAGAS_EVAL_EMBED_MODEL,
        model_kwargs={"device": DEVICE},
        encode_kwargs={"normalize_embeddings": True},
    )
    evaluator_embeddings = LangchainEmbeddingsWrapper(eval_embeddings)
    return evaluator_llm, evaluator_embeddings


def evaluate_with_ragas(results_df: pd.DataFrame) -> pd.DataFrame:
    from ragas import evaluate
    from ragas.run_config import RunConfig

    evaluator_llm, evaluator_embeddings = build_ragas_local_wrappers()
    rows = results_df.to_dict("records")
    run_config = RunConfig(
        timeout=RAGAS_TIMEOUT,
        max_workers=RAGAS_MAX_WORKERS,
        max_retries=RAGAS_MAX_RETRIES,
        seed=SEED,
    )

    print("RAGAS reel : evaluation des reponses pour les 3 systemes...")
    answer_dataset = make_ragas_dataset(rows)
    answer_metrics = build_ragas_metrics("answer")
    answer_scores = evaluate(
        answer_dataset,
        metrics=answer_metrics,
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        run_config=run_config,
        raise_exceptions=False,
    ).to_pandas()

    print("RAGAS reel : evaluation des contextes pour les modes RAG...")
    rag_rows = [r for r in rows if r["system"] != "Generation only"]
    rag_dataset = make_ragas_dataset(rag_rows)
    rag_metrics = build_ragas_metrics("rag")
    rag_scores = evaluate(
        rag_dataset,
        metrics=rag_metrics,
        llm=evaluator_llm,
        embeddings=evaluator_embeddings,
        run_config=run_config,
        raise_exceptions=False,
    ).to_pandas()

    # Selon la version de RAGAS, `to_pandas()` peut ne garder que les colonnes
    # de metriques. On rattache donc les identifiants par position, ce qui
    # evite le KeyError: "['system', 'language'] not in index".
    answer_meta = pd.DataFrame([
        {"system": r["system"], "language": r["language"], "user_input": r["question"]}
        for r in rows
    ])
    rag_meta = pd.DataFrame([
        {"system": r["system"], "language": r["language"], "user_input": r["question"]}
        for r in rag_rows
    ])

    for col in ["system", "language", "user_input"]:
        if col in answer_scores.columns:
            answer_scores = answer_scores.drop(columns=[col])
        if col in rag_scores.columns:
            rag_scores = rag_scores.drop(columns=[col])

    answer_scores = pd.concat(
        [answer_meta.reset_index(drop=True), answer_scores.reset_index(drop=True)],
        axis=1,
    )
    rag_scores = pd.concat(
        [rag_meta.reset_index(drop=True), rag_scores.reset_index(drop=True)],
        axis=1,
    )

    base_cols = ["system", "language", "user_input"]
    ignored = {
        "row_id", "question", "answer", "contexts", "ground_truth",
        "user_input", "response", "retrieved_contexts", "reference", "language",
    }
    answer_metric_cols = [
        c for c in answer_scores.columns
        if c not in ignored and c not in base_cols and pd.api.types.is_numeric_dtype(answer_scores[c])
    ]
    rag_metric_cols = [
        c for c in rag_scores.columns
        if c not in ignored and c not in base_cols and pd.api.types.is_numeric_dtype(rag_scores[c])
    ]

    answer_part = answer_scores[base_cols + answer_metric_cols].copy()
    rag_part = rag_scores[base_cols + rag_metric_cols].copy()
    merged = answer_part.merge(rag_part, on=base_cols, how="left")

    out_path = WORK_DIR / "ragas_true_scores.csv"
    merged.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Scores RAGAS reels sauvegardes : {out_path}")
    return merged


In [33]:
# ============================================================

def summarize_and_plot(results_df: pd.DataFrame, ragas_df: pd.DataFrame) -> pd.DataFrame:
    merged = results_df.merge(
        ragas_df,
        left_on=["system", "language", "question"],
        right_on=["system", "language", "user_input"],
        how="left",
    )

    preferred_metrics = [
        "answer_relevancy",
        "answer_correctness",
        "faithfulness",
        "context_recall",
        "context_precision",
    ]
    metric_cols = [c for c in preferred_metrics if c in merged.columns]
    numeric_cols = metric_cols + ["n_candidates", "n_contexts", "top_score", "seconds_generation"]
    summary = merged.groupby("system")[numeric_cols].mean(numeric_only=True).round(3)

    summary_path = WORK_DIR / "ablation_summary_ragas_true.csv"
    summary.to_csv(summary_path, encoding="utf-8-sig")
    print("\nResume final :")
    print(summary.to_string())
    print(f"Resume sauvegarde : {summary_path}")

    if metric_cols:
        ax = summary[metric_cols].plot(kind="bar", figsize=(12, 5), rot=0)
        ax.set_title("Ablation : Generation only vs RAG sans re-ranker vs RAG + re-ranker")
        ax.set_ylabel("Score RAGAS moyen")
        ax.set_ylim(0, 1)
        ax.grid(axis="y", alpha=0.25)
        plt.tight_layout()
        plot_path = WORK_DIR / "ablation_ragas_true.png"
        plt.savefig(plot_path, dpi=160, bbox_inches="tight")
        print(f"Figure sauvegardee : {plot_path}")

    return summary


In [ ]:
pip install langchain-google-vertexai
pip install git+https://github.com/GoldSharon/ragas.git


In [ ]:
# ============================================================

def main() -> None:
    print("=" * 90)
    print("Hadith RAG Kaggle - version corrigee complete")
    print("=" * 90)

    save_justification()

    df = load_hadith_dataset()
    save_data_diagnostics(df)
    chunks = build_hadith_chunks(df)
    chunk_map = {c["uid"]: c for c in chunks}
    chunk_id_to_uid = {c["chunk_id"]: c["uid"] for c in chunks}

    graph, chapter_to_uids, uid_to_chapter = build_chapter_graph(chunks)
    print(f"Graphe documentaire : {graph.number_of_nodes():,} noeuds / {graph.number_of_edges():,} aretes")
    print(f"Groupes chapitre    : {len(chapter_to_uids):,}")
    print(f"Chunks              : {len(chunks):,}")

    idf_func = make_token_idf(chunks)

    embed_model = load_embedding_model()
    collection = get_or_build_chroma_collection(embed_model, chunks)
    reranker = load_cross_encoder_reranker()

    ablation_df = prepare_ablation_rows(
        embed_model=embed_model,
        collection=collection,
        chunk_map=chunk_map,
        chunk_id_to_uid=chunk_id_to_uid,
        reranker=reranker,
        idf_func=idf_func,
    )

    print("Liberation memoire GPU avant chargement du LLM...")
    del embed_model
    del reranker
    clear_memory()

    tokenizer, model = load_generation_llm()
    results_df = generate_answers(ablation_df, tokenizer, model)

    print("Liberation du LLM de generation avant RAGAS...")
    del tokenizer
    del model
    clear_memory()

    ragas_df = evaluate_with_ragas(results_df)
    summarize_and_plot(results_df, ragas_df)

    print("\nVerification finale attendue :")
    print("- ChromaDB utilise : oui")
    print(f"- Similarite Chroma : cosine")
    print(f"- Embedding retrieval : {EMBED_MODEL_NAME}")
    print(f"- Re-ranker reel : {RERANK_MODEL_NAME}")
    print("- Mode re-ranker dans les resultats : cross_encoder_plus_hybrid")
    print(f"- LLM local Kaggle : {HF_LLM_MODEL}")
    print("- Ablations : Generation only / RAG sans re-ranker / RAG + re-ranker")
    print("- Evaluation : RAGAS reel, pas de proxy metrics")
    print("\nFichiers dans /kaggle/working :")
    print("- justification_choix_architecture_metriques.md")
    print("- all_related_hadiths_per_question.json")
    print("- ablation_generation_vs_rag_vs_reranker_raw.csv")
    print("- ragas_true_scores.csv")
    print("- ablation_summary_ragas_true.csv")
    print("- ablation_ragas_true.png")


if __name__ == "__main__":
    main()


In [ ]:
# ==========================================
# CELLULE COMPATIBILITE BACKEND POUR GRADIO
# A lancer AVANT la cellule Gradio
# ==========================================

import os, sys, subprocess, contextlib, gc, io, re, random, unicodedata
from pathlib import Path
from collections import defaultdict
from typing import Any

os.environ.setdefault("MAX_HADITHS", "5000")
os.environ.setdefault("TOP_K_RETRIEVAL", "50")
os.environ.setdefault("RERANK_CANDIDATES", "12")
os.environ.setdefault("TOP_CONTEXTS_FOR_LLM", "5")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

try:
    import chromadb, torch
    import numpy as np
    import pandas as pd
    from datasets import load_dataset
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "chromadb", "datasets", "FlagEmbedding", "transformers",
                    "accelerate", "sentencepiece", "protobuf", "pandas", "numpy"], check=True)
    import chromadb, torch
    import numpy as np
    import pandas as pd
    from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
DATASET_NAME = "gurgutan/sunnah_ar_en_dataset"

MAX_HADITHS = int(os.environ.get("MAX_HADITHS", "5000"))
EMBED_MODEL_NAME = os.environ.get("EMBED_MODEL_NAME", "BAAI/bge-m3")
RERANK_MODEL_NAME = os.environ.get("RERANK_MODEL_NAME", "BAAI/bge-reranker-v2-m3")
HF_LLM_MODEL = os.environ.get("HF_LLM_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")

CHROMA_DIR = str(WORK_DIR / "chroma_hadith_bge_m3_cosine")
COLLECTION_NAME = os.environ.get("COLLECTION_NAME", f"hadith_bge_m3_{MAX_HADITHS or 'full'}")

TOP_K_RETRIEVAL = int(os.environ.get("TOP_K_RETRIEVAL", "50"))
RERANK_CANDIDATES = int(os.environ.get("RERANK_CANDIDATES", "12"))
TOP_CONTEXTS_FOR_LLM = int(os.environ.get("TOP_CONTEXTS_FOR_LLM", "5"))
MIN_RETRIEVAL_SIMILARITY = float(os.environ.get("MIN_RETRIEVAL_SIMILARITY", "0.18"))

EMBED_BATCH_SIZE = int(os.environ.get("EMBED_BATCH_SIZE", "24"))
RERANK_BATCH_SIZE = int(os.environ.get("RERANK_BATCH_SIZE", "2"))
RERANK_MAX_LENGTH = int(os.environ.get("RERANK_MAX_LENGTH", "384"))
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", "360"))
MAX_CONTEXT_CHARS = int(os.environ.get("MAX_CONTEXT_CHARS", "10000"))

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
QUIET_MODELS = True

AR_NORMALIZE_TABLE = str.maketrans({
    "أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا",
    "ى": "ي", "ة": "ه", "ؤ": "و", "ئ": "ي",
})

STOPWORDS = {
    "the", "and", "or", "to", "of", "in", "a", "an", "is", "are",
    "what", "how", "did", "does", "about", "with", "from", "for",
    "على", "عن", "في", "ما", "هي", "هو", "من", "الى", "إلى", "قال",
    "النبي", "رسول", "الله", "صلى", "عليه", "وسلم",
}

AR_TO_EN_TERMS = {
    "الصيام": "fasting sawm ramadan",
    "صيام": "fasting sawm ramadan",
    "رمضان": "ramadan fasting forgiveness reward",
    "الصلاة": "prayer salat salah worship",
    "صلاة": "prayer salat salah worship",
    "الزكاة": "zakat charity",
    "الحج": "hajj pilgrimage",
    "أركان": "pillars islam five",
    "الصدق": "truthfulness honesty",
    "الأمانة": "trust honesty amanah",
    "الوالدين": "parents mother father kindness",
    "بر": "kindness honoring parents",
    "الجيران": "neighbors kindness",
    "النفاق": "hypocrisy hypocrite lies promise trust",
    "المنافق": "hypocrite lies promise trust",
    "فضائل": "virtues merits reward",
    "فضل": "virtue merit reward",
    "حكم": "ruling obligation required",
}

EN_SYNONYMS = {
    "prayer": "salat salah worship",
    "salat": "prayer salah worship",
    "fasting": "sawm ramadan forgiveness reward",
    "ramadan": "fasting sawm forgiveness reward",
    "parents": "mother father honoring kindness",
    "neighbor": "neighbour neighbor kindness",
    "neighbors": "neighbour neighbor kindness",
    "hypocrite": "munafiq lies promise trust",
    "truth": "truthfulness honesty amanah",
    "charity": "sadaqah zakat",
    "pillars": "five islam shahada prayer zakat fasting hajj",
}

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def is_arabic(text: str) -> bool:
    text = str(text or "")
    arabic_chars = sum(1 for c in text if "\u0600" <= c <= "\u06FF")
    return arabic_chars > max(1, len(text)) * 0.25

def normalize_text(text: str) -> str:
    text = str(text or "").lower()
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = text.translate(AR_NORMALIZE_TABLE)
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def score_tokens(text: str) -> set[str]:
    return {t for t in normalize_text(text).split() if len(t) > 2 and t not in STOPWORDS}

def expand_query(query: str) -> str:
    expanded = str(query)
    if is_arabic(query):
        for ar, en in AR_TO_EN_TERMS.items():
            if ar in query:
                expanded += " " + en
    else:
        q = query.lower()
        for term, syns in EN_SYNONYMS.items():
            if term in q:
                expanded += " " + syns
    return expanded

def l2_normalize(vectors):
    arr = np.asarray(vectors, dtype=np.float32)
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)
    norms = np.linalg.norm(arr, axis=1, keepdims=True)
    return arr / np.maximum(norms, 1e-12)

def load_hadith_dataset():
    ds = load_dataset(DATASET_NAME)
    split = list(ds.keys())[0]
    df = ds[split].to_pandas()

    cols = [
        "hadith_uid", "hadith_text_en", "hadith_text_ar",
        "hadith_narrator_en", "hadith_chapter_name_en",
        "hadith_chapter_name_ar", "book_title_en",
    ]
    for col in cols:
        df[col] = df[col].fillna("").astype(str).str.strip()

    df = df[
        (df["hadith_text_en"].str.len() > 10)
        | (df["hadith_text_ar"].str.len() > 10)
    ].reset_index(drop=True)

    if MAX_HADITHS > 0 and len(df) > MAX_HADITHS:
        df = df.sample(n=MAX_HADITHS, random_state=SEED).sort_index().reset_index(drop=True)

    return df

def build_hadith_chunks(df):
    chunks = []
    for row in df.itertuples(index=False):
        uid = str(row.hadith_uid)
        text_combined = (
            f"Book: {row.book_title_en}\n"
            f"Chapter EN: {row.hadith_chapter_name_en}\n"
            f"Chapter AR: {row.hadith_chapter_name_ar}\n"
            f"Narrator: {row.hadith_narrator_en}\n"
            f"English Hadith: {row.hadith_text_en}\n"
            f"Arabic Hadith: {row.hadith_text_ar}"
        )
        chunks.append({
            "chunk_id": f"h__{uid}",
            "uid": uid,
            "book": str(row.book_title_en),
            "chapter_en": str(row.hadith_chapter_name_en),
            "chapter_ar": str(row.hadith_chapter_name_ar),
            "narrator": str(row.hadith_narrator_en),
            "text_en": str(row.hadith_text_en),
            "text_ar": str(row.hadith_text_ar),
            "text_combined": text_combined,
        })
    return chunks

def make_token_idf(chunks):
    df_counts = defaultdict(int)
    for c in chunks:
        for t in score_tokens(c["text_combined"]):
            df_counts[t] += 1
    n_docs = len(chunks)

    def idf(token):
        return float(np.log((n_docs + 1) / (df_counts.get(token, 0) + 1)) + 1.0)

    return idf

def weighted_overlap(query_terms, doc_terms, idf_func):
    if not query_terms:
        return 0.0
    denom = sum(idf_func(t) for t in query_terms)
    return 0.0 if denom <= 0 else sum(idf_func(t) for t in query_terms & doc_terms) / denom

def lexical_score(query, h, idf_func):
    q_terms = score_tokens(expand_query(query))
    doc_terms = score_tokens(
        f"{h.get('book','')} {h.get('chapter_en','')} {h.get('chapter_ar','')} "
        f"{h.get('narrator','')} {h.get('text_en','')} {h.get('text_ar','')}"
    )
    return weighted_overlap(q_terms, doc_terms, idf_func)

def load_embedding_model():
    from FlagEmbedding import BGEM3FlagModel
    print(f"Chargement embedding : {EMBED_MODEL_NAME}")
    return BGEM3FlagModel(EMBED_MODEL_NAME, use_fp16=torch.cuda.is_available())

def bge_dense_encode(embed_model, texts, batch_size=EMBED_BATCH_SIZE):
    single = isinstance(texts, str)
    text_list = [texts] if single else list(texts)

    stdout_buf, stderr_buf = io.StringIO(), io.StringIO()
    with contextlib.redirect_stdout(stdout_buf), contextlib.redirect_stderr(stderr_buf):
        try:
            output = embed_model.encode(text_list, return_dense=True, batch_size=batch_size)
        except TypeError:
            output = embed_model.encode(text_list, return_dense=True)

    vectors = output["dense_vecs"] if isinstance(output, dict) else output
    vectors = l2_normalize(vectors)
    return vectors[0] if single else vectors

def get_or_build_chroma_collection(embed_model, chunks):
    client = chromadb.PersistentClient(path=CHROMA_DIR)

    try:
        collection = client.get_collection(COLLECTION_NAME)
        print(f"Collection Chroma chargée : {COLLECTION_NAME} ({collection.count():,})")
        return collection
    except Exception:
        print("Collection Chroma absente, création de l'index...")
        collection = client.create_collection(
            name=COLLECTION_NAME,
            metadata={"hnsw:space": "cosine"},
        )

    batch_size = 96
    for start in range(0, len(chunks), batch_size):
        batch = chunks[start:start + batch_size]
        ids = [c["chunk_id"] for c in batch]
        docs = [c["text_combined"] for c in batch]
        metas = [
            {
                "uid": c["uid"],
                "book": c["book"],
                "chapter_en": c["chapter_en"],
                "chapter_ar": c["chapter_ar"],
                "narrator": c["narrator"],
            }
            for c in batch
        ]
        emb = bge_dense_encode(embed_model, docs).tolist()
        collection.add(ids=ids, documents=docs, metadatas=metas, embeddings=emb)
        print(f"Index Chroma : {min(start + batch_size, len(chunks)):,}/{len(chunks):,}")
        clear_memory()

    return collection

def load_cross_encoder_reranker():
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    print(f"Chargement re-ranker : {RERANK_MODEL_NAME}")

    tok = AutoTokenizer.from_pretrained(RERANK_MODEL_NAME, trust_remote_code=True)
    mdl = AutoModelForSequenceClassification.from_pretrained(
        RERANK_MODEL_NAME,
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    ).to(DEVICE)
    mdl.eval()
    return {"tokenizer": tok, "model": mdl}

def candidate_text_for_reranker(query, h):
    primary = h.get("text_ar", "") if is_arabic(query) else h.get("text_en", "")
    secondary = h.get("text_en", "") if is_arabic(query) else h.get("text_ar", "")
    meta = f"Book: {h.get('book','')}\nChapter: {h.get('chapter_en','')}\nNarrator: {h.get('narrator','')}"
    return f"{meta}\n{primary}\n{secondary}"[:1200]

def manual_cross_encoder_scores(query, candidates, reranker):
    tok = reranker["tokenizer"]
    mdl = reranker["model"]
    device = next(mdl.parameters()).device
    scores = []

    for start in range(0, len(candidates), RERANK_BATCH_SIZE):
        batch = candidates[start:start + RERANK_BATCH_SIZE]
        docs = [candidate_text_for_reranker(query, h) for h in batch]
        enc = tok(
            [query] * len(docs),
            docs,
            padding=True,
            truncation=True,
            max_length=RERANK_MAX_LENGTH,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            out = mdl(**enc)
            logits = out.logits
            batch_scores = logits[:, -1] if logits.ndim == 2 and logits.shape[1] > 1 else logits.reshape(-1)
        scores.extend(batch_scores.float().detach().cpu().tolist())
        clear_memory()

    return scores

def retrieve_related_hadiths(query, embed_model, collection, chunk_map, chunk_id_to_uid,
                             top_k=TOP_K_RETRIEVAL, min_similarity=MIN_RETRIEVAL_SIMILARITY):
    q_emb = bge_dense_encode(embed_model, expand_query(query)).reshape(-1).tolist()
    res = collection.query(
        query_embeddings=[q_emb],
        n_results=int(top_k),
        include=["metadatas", "distances"],
    )

    candidates = []
    for rank, chunk_id in enumerate(res["ids"][0], 1):
        uid = chunk_id_to_uid.get(chunk_id) or res["metadatas"][0][rank - 1].get("uid")
        if uid not in chunk_map:
            continue
        similarity = 1.0 - float(res["distances"][0][rank - 1])
        if similarity < min_similarity:
            continue
        item = chunk_map[uid].copy()
        item["rank_retrieval"] = rank
        item["score_retrieval"] = round(similarity, 4)
        candidates.append(item)

    return candidates

def rag_without_reranker(candidates, top_n=TOP_CONTEXTS_FOR_LLM):
    selected = []
    for idx, h in enumerate(candidates[:int(top_n)], 1):
        item = h.copy()
        item["score_final"] = item.get("score_retrieval", 0.0)
        item["rerank_mode"] = "vector_only_no_reranker"
        item["rank_final"] = idx
        selected.append(item)
    return selected

def rag_with_cross_encoder_reranker(query, candidates, reranker, idf_func, top_n=TOP_CONTEXTS_FOR_LLM):
    subset = candidates[:RERANK_CANDIDATES]
    if not subset:
        return []

    raw_scores = np.asarray(manual_cross_encoder_scores(query, subset, reranker), dtype=np.float32)
    if len(raw_scores) > 1 and float(raw_scores.max()) > float(raw_scores.min()):
        cross_scores = (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())
    else:
        cross_scores = np.ones_like(raw_scores) * 0.5

    ranked = []
    for h, cross in zip(subset, cross_scores.tolist()):
        lex = lexical_score(query, h, idf_func)
        final = 0.78 * float(cross) + 0.15 * float(h.get("score_retrieval", 0.0)) + 0.07 * lex
        item = h.copy()
        item["score_cross_encoder"] = round(float(cross), 4)
        item["score_lexical"] = round(float(lex), 4)
        item["score_final"] = round(float(final), 4)
        item["rerank_mode"] = "cross_encoder_plus_hybrid"
        ranked.append(item)

    return sorted(ranked, key=lambda x: x["score_final"], reverse=True)[:int(top_n)]

def load_generation_llm():
    from transformers import AutoModelForCausalLM, AutoTokenizer

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    print(f"Chargement LLM : {HF_LLM_MODEL}")

    tok = AutoTokenizer.from_pretrained(HF_LLM_MODEL, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        HF_LLM_MODEL,
        torch_dtype=dtype,
        device_map="auto" if torch.cuda.is_available() else None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    if not torch.cuda.is_available():
        mdl = mdl.to("cpu")
    mdl.eval()
    return tok, mdl

def generate_with_llm(tokenizer, model, system, user_prompt):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user_prompt},
    ]

    if getattr(tokenizer, "chat_template", None):
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = f"System: {system}\nUser: {user_prompt}\nAssistant:"

    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=4096)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen = output[0][inputs["input_ids"].shape[-1]:]
    answer = tokenizer.decode(gen, skip_special_tokens=True).strip()
    clear_memory()
    return answer

def build_context_block(row):
    evidence = row.get("evidence", [])
    language = row.get("language", "ar" if is_arabic(row.get("question", "")) else "en")
    blocks = []
    total = 0

    for i, h in enumerate(evidence, 1):
        primary = h.get("text_ar", "") if language == "ar" else h.get("text_en", "")
        secondary = h.get("text_en", "") if language == "ar" else h.get("text_ar", "")
        block = (
            f"[H{i}]\n"
            f"Book: {h.get('book','')}\n"
            f"Chapter EN: {h.get('chapter_en','')}\n"
            f"Chapter AR: {h.get('chapter_ar','')}\n"
            f"Narrator: {h.get('narrator','')}\n"
            f"Primary text:\n{primary}\n"
            f"Other language text:\n{secondary}"
        )
        if blocks and total + len(block) > MAX_CONTEXT_CHARS:
            break
        blocks.append(block)
        total += len(block)

    return "\n\n".join(blocks)

print("Chargement backend Gradio...")

if "chunks" not in globals() or "chunk_map" not in globals() or "chunk_id_to_uid" not in globals():
    df_ui = load_hadith_dataset()
    chunks = build_hadith_chunks(df_ui)
    chunk_map = {c["uid"]: c for c in chunks}
    chunk_id_to_uid = {c["chunk_id"]: c["uid"] for c in chunks}
    print(f"Chunks chargés : {len(chunks):,}")

if "idf_func" not in globals():
    idf_func = make_token_idf(chunks)

if "embed_model" not in globals():
    embed_model = load_embedding_model()

if "collection" not in globals():
    collection = get_or_build_chroma_collection(embed_model, chunks)

if "reranker" not in globals():
    reranker = load_cross_encoder_reranker()

if "tokenizer" not in globals() or "model" not in globals():
    tokenizer, model = load_generation_llm()

print("Backend Gradio prêt. Tu peux relancer la cellule interface.")

In [ ]:
# =========================
# GRADIO UI PRO - VERSION HTML RENDER
# =========================

import html, time
import gradio as gr

def ui_build_prompt(question, contexts):
    context = build_context_block({
        "question": question,
        "language": "ar" if is_arabic(question) else "en",
        "evidence": contexts,
    })

    if is_arabic(question):
        system = "أنت مساعد RAG متخصص في الأحاديث. استخدم النصوص المسترجعة فقط واستشهد مثل [H1]."
        prompt = (
            "استخدم الأحاديث التالية فقط للإجابة. لا تضف حديثا غير موجود في السياق.\n\n"
            f"{context}\n\n"
            f"السؤال: {question}\n\n"
            "الإجابة المختصرة مع الاستشهادات:"
        )
    else:
        system = "You are a Hadith RAG assistant. Use only retrieved evidence and cite IDs like [H1]."
        prompt = (
            "Use ONLY the retrieved hadiths below. Do not add hadiths not present in the context.\n\n"
            f"{context}\n\n"
            f"Question: {question}\n\n"
            "Concise answer with citations:"
        )

    return system, prompt


def ui_render_sources(hadiths):
    if not hadiths:
        return "<div class='empty'>Aucune source récupérée.</div>"

    cards = []
    for i, h in enumerate(hadiths, 1):
        score = h.get("score_final", h.get("score_retrieval", 0.0))
        cross = h.get("score_cross_encoder", None)
        cross_txt = f" | cross-encoder {cross:.3f}" if cross is not None else ""

        cards.append(f"""
<details class="source-card" open>
  <summary>
    <span class="source-id">H{i}</span>
    <span>{html.escape(h.get("book", "Unknown"))}</span>
    <span class="score">score {score:.3f}{cross_txt}</span>
  </summary>

  <div class="source-meta">
    <b>Chapter:</b> {html.escape(h.get("chapter_en", ""))}<br>
    <b>Narrator:</b> {html.escape(h.get("narrator", ""))}<br>
    <b>Mode:</b> {html.escape(h.get("rerank_mode", ""))}
  </div>

  <div class="hadith-text en">
    <div class="label">English text</div>
    {html.escape(h.get("text_en", ""))}
  </div>

  <div class="hadith-text ar" dir="rtl">
    <div class="label">النص العربي</div>
    {html.escape(h.get("text_ar", ""))}
  </div>
</details>
""")
    return "\n".join(cards)


def ui_answer_html(question, use_reranker, top_k, rerank_candidates, final_sources):
    if not question or not question.strip():
        return "<div class='empty'>Pose une question pour commencer.</div>"

    question = question.strip()
    t0 = time.time()

    try:
        candidates = retrieve_related_hadiths(
            question,
            embed_model=embed_model,
            collection=collection,
            chunk_map=chunk_map,
            chunk_id_to_uid=chunk_id_to_uid,
            top_k=int(top_k),
            min_similarity=MIN_RETRIEVAL_SIMILARITY,
        )

        if use_reranker:
            evidence = rag_with_cross_encoder_reranker(
                question,
                candidates,
                reranker=reranker,
                idf_func=idf_func,
                top_n=int(final_sources),
            )
            mode = "RAG + BGE cross-encoder re-ranker"
        else:
            evidence = rag_without_reranker(candidates, top_n=int(final_sources))
            mode = "RAG sans re-ranker"

        system, prompt = ui_build_prompt(question, evidence)
        answer = generate_with_llm(tokenizer, model, system, prompt)

        elapsed = time.time() - t0
        sources_html = ui_render_sources(evidence)
        direction = "rtl" if is_arabic(question) else "ltr"

        return f"""
<div class="shell">
  <div class="question-card" dir="{direction}">
    <div class="eyebrow">Question</div>
    <div>{html.escape(question)}</div>
  </div>

  <div class="answer-card" dir="{direction}">
    <div class="answer-topline">
      <span class="pill">{html.escape(mode)}</span>
      <span class="latency">{elapsed:.1f}s</span>
    </div>
    <h2>Réponse</h2>
    <div class="answer-text">{html.escape(answer).replace(chr(10), "<br>")}</div>
  </div>

  <div class="stats-grid">
    <div><b>{len(candidates)}</b><span>candidats récupérés</span></div>
    <div><b>{len(evidence)}</b><span>sources finales</span></div>
    <div><b>{evidence[0].get("rerank_mode", "none") if evidence else "none"}</b><span>mode</span></div>
  </div>

  <h2 class="section-title">Sources complètes</h2>
  {sources_html}
</div>
"""

    except Exception as e:
        return f"""
<div class="error-card">
  <h2>Erreur</h2>
  <p>{html.escape(str(e))}</p>
</div>
"""


custom_css = """
.gradio-container {
    max-width: 1240px !important;
    margin: auto !important;
    background: #f6f7fb;
    font-family: Inter, ui-sans-serif, system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif;
}

.hero {
    background: white;
    border: 1px solid #e5e7eb;
    border-radius: 16px;
    padding: 22px 24px;
    margin: 12px 0 18px 0;
    box-shadow: 0 14px 34px rgba(15, 23, 42, 0.07);
}

.hero h1 {
    margin: 0 0 8px 0;
    font-size: 34px;
    color: #0f172a;
}

.hero p {
    margin: 0;
    color: #475569;
    font-size: 15px;
}

.shell {
    padding: 4px 0 20px 0;
}

.question-card,
.answer-card {
    background: white;
    border: 1px solid #e5e7eb;
    border-radius: 14px;
    padding: 18px 20px;
    margin: 12px 0;
    box-shadow: 0 12px 28px rgba(15, 23, 42, 0.06);
}

.eyebrow {
    color: #0f766e;
    font-size: 12px;
    font-weight: 800;
    text-transform: uppercase;
    margin-bottom: 8px;
}

.answer-topline {
    display: flex;
    justify-content: space-between;
    align-items: center;
    margin-bottom: 10px;
}

.pill {
    background: #0f766e;
    color: white;
    padding: 5px 10px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: 800;
}

.latency {
    color: #64748b;
    font-size: 13px;
}

.answer-card h2,
.section-title {
    font-size: 20px;
    margin: 8px 0 12px 0;
    color: #0f172a;
}

.answer-text {
    color: #1f2937;
    line-height: 1.75;
    font-size: 15.5px;
}

.stats-grid {
    display: grid;
    grid-template-columns: repeat(3, minmax(0, 1fr));
    gap: 10px;
    margin: 12px 0 18px 0;
}

.stats-grid div {
    background: white;
    border: 1px solid #e2e8f0;
    border-radius: 12px;
    padding: 12px;
}

.stats-grid b {
    display: block;
    color: #111827;
    font-size: 18px;
    margin-bottom: 4px;
}

.stats-grid span {
    color: #64748b;
    font-size: 12px;
}

.source-card {
    background: white;
    border: 1px solid #dbe3ef;
    border-radius: 12px;
    padding: 12px 14px;
    margin: 12px 0;
    box-shadow: 0 8px 18px rgba(15, 23, 42, 0.045);
}

.source-card summary {
    cursor: pointer;
    font-weight: 800;
    color: #0f172a;
    display: flex;
    gap: 10px;
    align-items: center;
}

.source-id {
    background: #1d4ed8;
    color: white;
    border-radius: 8px;
    padding: 3px 8px;
    font-size: 12px;
}

.score {
    margin-left: auto;
    color: #475569;
    font-size: 12px;
    font-weight: 700;
}

.source-meta {
    margin: 12px 0;
    color: #475569;
    font-size: 13px;
    line-height: 1.5;
}

.hadith-text {
    border-radius: 10px;
    padding: 12px;
    margin-top: 10px;
    line-height: 1.65;
    color: #1e293b;
    background: #f8fafc;
}

.hadith-text.ar {
    background: #fff7ed;
    text-align: right;
    font-size: 16px;
}

.label {
    font-weight: 800;
    color: #334155;
    margin-bottom: 6px;
    font-size: 13px;
}

.error-card {
    background: #fee2e2;
    border: 1px solid #fecaca;
    color: #991b1b;
    border-radius: 12px;
    padding: 16px;
}

.empty {
    background: white;
    border: 1px dashed #cbd5e1;
    color: #64748b;
    border-radius: 12px;
    padding: 18px;
}
"""

with gr.Blocks(css=custom_css, title="Hadith RAG Assistant") as demo:
    gr.HTML("""
    <div class="hero">
      <h1>Hadith RAG Assistant</h1>
      <p>Recherche bilingue avec ChromaDB, BGE-M3, re-ranker BGE cross-encoder et génération sourcée.</p>
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=3):
            question = gr.Textbox(
                label="Question",
                placeholder="Exemple : What are the virtues of fasting in Ramadan? / ما حكم الصيام في رمضان؟",
                lines=3,
            )

            with gr.Row():
                submit = gr.Button("Rechercher", variant="primary")
                clear = gr.Button("Effacer")

            output = gr.HTML(
                value="<div class='empty'>Pose une question pour afficher la réponse et les sources.</div>",
                label="Résultat",
            )

        with gr.Column(scale=1):
            use_reranker = gr.Checkbox(value=True, label="Activer le re-ranker BGE")
            top_k = gr.Slider(10, 100, value=TOP_K_RETRIEVAL, step=5, label="Candidats récupérés")
            rerank_candidates = gr.Slider(4, 30, value=RERANK_CANDIDATES, step=1, label="Candidats re-rankés")
            final_sources = gr.Slider(1, 8, value=TOP_CONTEXTS_FOR_LLM, step=1, label="Sources finales")

            gr.Examples(
                examples=[
                    "What are the virtues of fasting in Ramadan?",
                    "What did the Prophet say about kindness to parents?",
                    "What are the signs of a hypocrite?",
                    "ما حكم الصيام في رمضان؟",
                    "ماذا قال النبي عن الصدق والأمانة؟",
                ],
                inputs=question,
                label="Exemples",
            )

    submit.click(
        ui_answer_html,
        [question, use_reranker, top_k, rerank_candidates, final_sources],
        output,
    )

    question.submit(
        ui_answer_html,
        [question, use_reranker, top_k, rerank_candidates, final_sources],
        output,
    )

    clear.click(
        lambda: ("", "<div class='empty'>Pose une question pour afficher la réponse et les sources.</div>"),
        None,
        [question, output],
    )

demo.launch(share=True, debug=False)